# Social Media Models — Training

Trains two models from `warehouse.fact_social_media_ml`:

1. **Engagement tier classifier** — Will a post get `Low / Medium / High` engagement?
2. **Donation conversion classifier** — Will a post lead to donations? (`0 / 1`)

Outputs: `engagement_model.sav`, `donation_model.sav` + metadata + metrics JSON files.

In [ ]:
import sys
sys.path.insert(0, '../jobs')

import json
from datetime import datetime, timezone

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from config import (
    ARTIFACTS_DIR,
    ENGAGEMENT_MODEL_PATH, ENGAGEMENT_METADATA_PATH, ENGAGEMENT_METRICS_PATH,
    DONATION_MODEL_PATH, DONATION_METADATA_PATH, DONATION_METRICS_PATH,
    WAREHOUSE_SCHEMA,
)
from utils_db import get_engine

print('Imports loaded.')

## Load data

In [ ]:
engine = get_engine(WAREHOUSE_SCHEMA)
df = pd.read_sql('SELECT * FROM warehouse.fact_social_media_ml', engine)
print(f'Loaded {len(df)} rows')

print(f'\nEngagement tier distribution:\n{df["engagement_tier"].value_counts().to_string()}')
print(f'\nHas donations distribution:\n{df["has_donations"].value_counts().to_string()}')

## Feature columns

In [ ]:
CAT_COLS = [
    'platform_encoded', 'post_type_encoded', 'media_type_encoded',
    'day_of_week_encoded', 'call_to_action_type_encoded',
    'content_topic_encoded', 'sentiment_tone_encoded',
]
NUM_COLS = [
    'has_call_to_action', 'features_resident_story', 'is_boosted',
    'post_hour', 'num_hashtags', 'mentions_count', 'caption_length',
    'boost_budget_php', 'follower_count_at_post',
]
FEATURE_COLS = CAT_COLS + NUM_COLS

CAT_CATEGORIES = [
    [0,1,2,3,4,5,6], [0,1,2,3,4,5], [0,1,2,3,4],
    [0,1,2,3,4,5,6], [0,1,2,3,4], [0,1,2,3,4,5,6,7,8], [0,1,2,3,4,5],
]

def make_pipeline():
    cat_transformer = Pipeline([('encoder', OrdinalEncoder(
        categories=CAT_CATEGORIES, handle_unknown='use_encoded_value', unknown_value=-1
    ))])
    num_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    preprocessor = ColumnTransformer([
        ('cat', cat_transformer, CAT_COLS),
        ('num', num_transformer, NUM_COLS),
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')),
    ])

X = df[FEATURE_COLS]
print(f'Feature matrix: {X.shape}')

## Model 1 — Engagement tier classifier

In [ ]:
y_eng = df['engagement_tier']
min_class = y_eng.value_counts().min()
stratify  = y_eng if min_class >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y_eng, test_size=0.25, random_state=42, stratify=stratify
)

eng_model = make_pipeline()
eng_model.fit(X_train, y_train)

y_pred   = eng_model.predict(X_test)
accuracy = float(accuracy_score(y_test, y_pred))
f1       = float(f1_score(y_test, y_pred, average='weighted'))
report   = classification_report(y_test, y_pred, output_dict=True)

print(f'Engagement — Accuracy: {accuracy:.3f} | Weighted F1: {f1:.3f}')
print(classification_report(y_test, y_pred))

In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(eng_model, str(ENGAGEMENT_MODEL_PATH))
with open(ENGAGEMENT_METADATA_PATH, 'w') as f:
    json.dump({'model_name': 'social_media_engagement_classifier', 'model_version': '1.0.0',
               'trained_at_utc': datetime.now(timezone.utc).isoformat(),
               'warehouse_table': 'fact_social_media_ml',
               'num_training_rows': int(len(X_train)), 'num_test_rows': int(len(X_test)),
               'label_col': 'engagement_tier', 'feature_cols': FEATURE_COLS,
               'cat_cols': CAT_COLS, 'num_cols': NUM_COLS,
               'classes': list(eng_model.classes_)}, f, indent=2)
with open(ENGAGEMENT_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy, 'weighted_f1': f1, 'classification_report': report}, f, indent=2)
print(f'Saved: {ENGAGEMENT_MODEL_PATH}')

## Model 2 — Donation conversion classifier

In [ ]:
y_don = df['has_donations'].astype(int)
min_class = y_don.value_counts().min()
stratify  = y_don if min_class >= 2 else None

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X, y_don, test_size=0.25, random_state=42, stratify=stratify
)

don_model = make_pipeline()
don_model.fit(X_train_d, y_train_d)

y_pred_d   = don_model.predict(X_test_d)
accuracy_d = float(accuracy_score(y_test_d, y_pred_d))
f1_d       = float(f1_score(y_test_d, y_pred_d, average='weighted'))
report_d   = classification_report(y_test_d, y_pred_d, output_dict=True)

print(f'Donation — Accuracy: {accuracy_d:.3f} | Weighted F1: {f1_d:.3f}')
print(classification_report(y_test_d, y_pred_d))

In [ ]:
joblib.dump(don_model, str(DONATION_MODEL_PATH))
with open(DONATION_METADATA_PATH, 'w') as f:
    json.dump({'model_name': 'social_media_donation_classifier', 'model_version': '1.0.0',
               'trained_at_utc': datetime.now(timezone.utc).isoformat(),
               'warehouse_table': 'fact_social_media_ml',
               'num_training_rows': int(len(X_train_d)), 'num_test_rows': int(len(X_test_d)),
               'label_col': 'has_donations', 'feature_cols': FEATURE_COLS,
               'cat_cols': CAT_COLS, 'num_cols': NUM_COLS,
               'classes': [int(c) for c in don_model.classes_]}, f, indent=2)
with open(DONATION_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy_d, 'weighted_f1': f1_d, 'classification_report': report_d}, f, indent=2)
print(f'Saved: {DONATION_MODEL_PATH}')